# ECHO Training Notebook
Trains Qwen2.5-7B to predict its own correctness using GRPO + OpenEnv

In [ ]:
# Install dependencies
!pip install -q "trl>=0.8.0" "peft" "transformers" "datasets" "huggingface_hub"
!pip install -q "openenv-core[core]>=0.2.0" || pip install -q git+https://github.com/meta-pytorch/OpenEnv.git
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import os
import requests
import json
import numpy as np
from huggingface_hub import login

# Authenticate
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Set in Colab secrets
if HF_TOKEN:
    login(HF_TOKEN)

# Connect to live ECHO environment on HuggingFace Spaces
ECHO_SPACE_URL = "https://vikaspandey582003-echo-ultimate.hf.space"

# Test connection
resp = requests.get(f"{ECHO_SPACE_URL}/health", timeout=10)
print(f"Space status: {resp.json()}")

In [ ]:
# Simple HTTP client for the ECHO environment
class EchoEnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")
    
    def reset(self):
        r = requests.post(f"{self.base_url}/reset", timeout=30)
        r.raise_for_status()
        return r.json()
    
    def step(self, response_text: str):
        # OpenEnv servers may accept either {"response": ...} or {"action": {"response": ...}}
        payloads = [
            {"response": response_text},
            {"action": {"response": response_text}},
        ]
        last_error = None
        for payload in payloads:
            try:
                r = requests.post(f"{self.base_url}/step", json=payload, timeout=30)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                last_error = e
        raise RuntimeError(f"Step request failed for all payload formats: {last_error}")
    
    def get_metrics(self):
        r = requests.get(f"{self.base_url}/metrics", timeout=10)
        r.raise_for_status()
        return r.json()

env = EchoEnvClient(ECHO_SPACE_URL)

# Test: reset and take a step
obs = env.reset()
print("Question:", obs.get("question", ""))
result = env.step("<confidence>70</confidence><answer>test answer</answer>")
print("Step response keys:", list(result.keys()))

In [ ]:
# Load model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

SYSTEM_PROMPT = """You are a calibrated AI assistant. For every question:
1. Think step-by-step (optional: use <think>...</think> tags)  
2. Output your confidence as an integer 0-100: <confidence>INTEGER</confidence>
3. Output your answer: <answer>YOUR ANSWER</answer>

Be honest about uncertainty. Overconfidence is penalized heavily."""

# Build dataset from ECHO environment
def build_training_dataset(n_samples=500):
    samples = []
    for _ in range(n_samples):
        obs = env.reset()
        question = obs.get("question", "")
        samples.append({
            "prompt": f"{SYSTEM_PROMPT}\n\nQuestion: {question}",
            "question": question,
        })
    return Dataset.from_list(samples)

print("Building training dataset from live environment...")
dataset = build_training_dataset(500)
print(f"Dataset size: {len(dataset)}")

In [ ]:
# GRPO reward function — calls live OpenEnv environment
ece_history = []
reward_history = []
confidence_eval_history = []
outcome_history = []

def _extract_step_values(result: dict):
    # Supports both flat and OpenEnv-shaped responses.
    obs = result.get("observation") or result.get("obs") or result.get("state") or {}
    info = result.get("info") or {}

    reward = result.get("reward", info.get("reward", obs.get("reward", 0.0)))
    ece = result.get("ece", info.get("ece", obs.get("ece", 0.5)))
    conf = result.get("confidence", obs.get("confidence", None))
    is_correct = result.get("is_correct", obs.get("is_correct", info.get("was_correct", None)))

    return float(reward), float(ece), conf, is_correct

def echo_reward_function(completions, prompts=None, **kwargs):
    """
    Reward function that evaluates each completion against the live ECHO environment.
    This is the core of GRPO training — the environment provides the reward signal.
    """
    rewards = []
    for i, completion in enumerate(completions):
        try:
            # Reset for each completion so reward is grounded to a fresh environment question.
            env.reset()

            # Each completion is evaluated by the running OpenEnv Space.
            result = env.step(completion)
            reward, ece, conf, is_correct = _extract_step_values(result)

            ece_history.append(ece)
            reward_history.append(reward)
            if conf is not None:
                confidence_eval_history.append(float(conf) / 100.0)
            if is_correct is not None:
                outcome_history.append(1.0 if bool(is_correct) else 0.0)
            rewards.append(reward)

        except Exception as e:
            print(f"Env step failed: {e}")
            rewards.append(-0.5)  # penalty for failed step

    return rewards

In [ ]:
# Configure GRPO training
training_args = GRPOConfig(
    output_dir="echo_grpo_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    report_to="none",
    max_completion_length=512,
    num_generations=4,  # GRPO group size
    temperature=0.8,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[echo_reward_function],
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("Starting GRPO training against live ECHO environment...")
trainer.train()
print("Training complete!")

In [ ]:
# Plot ECE curve, reward curve, and reliability diagram
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

# ECE curve
if ece_history:
    window = 50
    smoothed = [np.mean(ece_history[max(0, i - window):i + 1]) for i in range(len(ece_history))]
    ax1.plot(ece_history, alpha=0.3, color='blue', label='Raw ECE')
    ax1.plot(smoothed, color='blue', linewidth=2, label='Smoothed ECE')
    ax1.axhline(y=0.15, color='green', linestyle='--', label='Good threshold (0.15)')
    ax1.axhline(y=0.20, color='orange', linestyle='--', label='Acceptable (0.20)')
    ax1.set_xlabel('Training Steps')
    ax1.set_ylabel('ECE (lower = better)')
    ax1.set_title('ECHO: ECE During GRPO Training')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# Reward curve
if reward_history:
    window = 50
    smoothed_r = [np.mean(reward_history[max(0, i - window):i + 1]) for i in range(len(reward_history))]
    ax2.plot(reward_history, alpha=0.3, color='green', label='Raw Reward')
    ax2.plot(smoothed_r, color='green', linewidth=2, label='Smoothed Reward')
    ax2.set_xlabel('Training Steps')
    ax2.set_ylabel('Reward')
    ax2.set_title('ECHO: Reward During GRPO Training')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# Reliability diagram
if confidence_eval_history and outcome_history and len(confidence_eval_history) == len(outcome_history):
    n_bins = 10
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    accs = []
    confs = []

    conf_arr = np.array(confidence_eval_history)
    out_arr = np.array(outcome_history)

    for i in range(n_bins):
        mask = (conf_arr >= bins[i]) & (conf_arr < bins[i + 1])
        if i == n_bins - 1:
            mask = (conf_arr >= bins[i]) & (conf_arr <= bins[i + 1])
        if np.any(mask):
            accs.append(float(np.mean(out_arr[mask])))
            confs.append(float(np.mean(conf_arr[mask])))
        else:
            accs.append(np.nan)
            confs.append(np.nan)

    ax3.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
    ax3.plot(bin_centers, accs, marker='o', linewidth=2, color='purple', label='Model')
    ax3.set_xlabel('Predicted confidence')
    ax3.set_ylabel('Empirical accuracy')
    ax3.set_title('Reliability Diagram')
    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    ax3.grid(True, alpha=0.3)
    ax3.legend()

plt.tight_layout()
plt.savefig("echo_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Final ECE: {ece_history[-1]:.4f}" if ece_history else "No ECE data")

In [ ]:
# Save and push adapter to HF Hub
model.save_pretrained("echo_lora_adapter")
tokenizer.save_pretrained("echo_lora_adapter")

from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="echo_lora_adapter",
    repo_id="Vikaspandey582003/echo-calibration-adapter",
    repo_type="model",
    commit_message="ECHO GRPO-trained calibration adapter - Hackathon submission",
)
print("Adapter pushed to HF Hub!")
print("Model: https://huggingface.co/Vikaspandey582003/echo-calibration-adapter")